# Meteorologia — distinguir resolução nativa e grade

Resolução real de **cinco minutos**, horizonte **t+5 minutos**. Os nomes dos arquivos foram mantidos por solicitação do usuário; o conteúdo foi migrado de diário para intradiário.

O alvo e a persistência (t−5 min) precisam ser originalmente medidos. Cada etapa de preenchimento causal usa até 15 minutos; no Brasil a combinação proxy + ffill pode atingir 30 minutos somente no histórico: nunca se interpola usando ponto posterior. Não se concatenam blocos interrompidos. A avaliação principal é diurna (elevação solar > 0); a avaliação 24h é complementar. Zero noturno não deve inflar a precisão.

As entradas principais são potência AC em p.u., hora local e dia do ano em seno/cosseno. O vento continua excluído. ERA5 nativa horária é auxiliar e não entra nos modelos principais. Estatísticas de normalização vêm apenas do treino australiano; alvos de validação/teste não ajustam escalas.


In [1]:
from pathlib import Path
import sys,json
import numpy as np
import pandas as pd
from IPython.display import display,Image,Markdown
RAIZ=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'src/energia_transfer').is_dir())
if str(RAIZ/'src') not in sys.path: sys.path.insert(0,str(RAIZ/'src'))
PASTA=RAIZ/'experimentos/transfer_learning_australia_brasil'
AU=RAIZ/'experimentos/australia_deep_learning'
REPORT=RAIZ/'reports/intradiario'
from energia_transfer.transfer_learning.dados import carregar,particao
_,CONFIG,CONTRATO=carregar(PASTA)
def exibir(t): display(t.round(5))


In [2]:
df=pd.read_pickle(RAIZ/'data/processed/brasil/base_5min_brasil.pkl')
exibir(df.head(15))
print('Alvos medidos:',df.Active_Power_original.notna().sum())
print('Históricos reconstruídos:',df.potencia_estimada.sum())
print('Grade:',df.index.to_series().diff().dropna().value_counts().to_dict())


Alvos medidos: 90023
Históricos reconstruídos: 12627
Grade: {Timedelta('0 days 00:05:00'): 105119}


,Active_Power_original,Active_Power,registro_original,diurno,inversores_medidos,inversores_proxy,inversores_zero_fisico,potencia_estimada,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,hora_sin,hora_cos,dia_ano_sin,dia_ano_cos,meteo_resolucao_nativa_min,meteo_idade_min,temperatura_estacao_original,ghi_estacao_original_escala_nao_validada
timestamp,,,,,,,,,,,,,,,,,,,
2025-01-01 00:00:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.00000,1.00000,0.0,1.0,60,0,NaN,NaN
2025-01-01 00:05:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.02181,0.99976,0.0,1.0,60,5,NaN,NaN
2025-01-01 00:10:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.04362,0.99905,0.0,1.0,60,10,NaN,NaN
2025-01-01 00:15:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.06540,0.99786,0.0,1.0,60,15,NaN,NaN
2025-01-01 00:20:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.08716,0.99619,0.0,1.0,60,20,NaN,NaN
2025-01-01 00:25:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.10887,0.99406,0.0,1.0,60,25,NaN,NaN
2025-01-01 00:30:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.13053,0.99144,0.0,1.0,60,30,NaN,NaN
2025-01-01 00:35:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.15212,0.98836,0.0,1.0,60,35,NaN,NaN
2025-01-01 00:40:00,0.0,0.0,True,False,8,0,0,False,NaN,NaN,NaN,0.17365,0.98481,0.0,1.0,60,40,NaN,NaN


## ERA5 não vira medição de cinco minutos

Temperatura/umidade/GHI ERA5 são horários e são mantidos para frente por no máximo 55 minutos, sem deslocar a radiação média da hora anterior para o início da hora (isso revelaria o futuro). A latência real de publicação não foi simulada. Estes dados são auxiliares, não entradas dos modelos principais.

A estação original tem longas interrupções, GHI com escala ainda não validada e não oferece a umidade necessária ao contrato meteorológico anterior. Não multiplicamos automaticamente GHI para fazê-lo coincidir com ERA5 e não interpolamos meses sem medições. Os faltantes da estação e da reanálise são relatados separadamente.


In [3]:
exibir(df[['Weather_Temperature_Celsius','Weather_Relative_Humidity','Global_Horizontal_Radiation','meteo_resolucao_nativa_min','meteo_idade_min','temperatura_estacao_original','ghi_estacao_original_escala_nao_validada']].head(24))
exibir(pd.read_csv(REPORT/'faltantes_brasil_resumo.csv'))


,Weather_Temperature_Celsius,Weather_Relative_Humidity,Global_Horizontal_Radiation,meteo_resolucao_nativa_min,meteo_idade_min,temperatura_estacao_original,ghi_estacao_original_escala_nao_validada
timestamp,,,,,,,
2025-01-01 00:00:00,NaN,NaN,NaN,60,0,NaN,NaN
2025-01-01 00:05:00,NaN,NaN,NaN,60,5,NaN,NaN
2025-01-01 00:10:00,NaN,NaN,NaN,60,10,NaN,NaN
2025-01-01 00:15:00,NaN,NaN,NaN,60,15,NaN,NaN
2025-01-01 00:20:00,NaN,NaN,NaN,60,20,NaN,NaN
2025-01-01 00:25:00,NaN,NaN,NaN,60,25,NaN,NaN
2025-01-01 00:30:00,NaN,NaN,NaN,60,30,NaN,NaN
2025-01-01 00:35:00,NaN,NaN,NaN,60,35,NaN,NaN
2025-01-01 00:40:00,NaN,NaN,NaN,60,40,NaN,NaN


,variavel,esperados,faltantes,faltantes_diurnos,maior_lacuna_min,faltantes_pct
0,Active_Power_original,105120,15097,5407,3270,14.36168
1,Global_Horizontal_Radiation,105120,288,157,1440,0.27397
2,Weather_Relative_Humidity,105120,288,157,1440,0.27397
3,Weather_Temperature_Celsius,105120,288,157,1440,0.27397
4,ghi_estacao_original_escala_nao_validada,105120,71737,35316,301065,68.24296
5,temperatura_estacao_original,105120,71737,35316,301065,68.24296
